# 05 — LLM as Judge

## Why this notebook exists

In **notebook 04** we built a regression gate using only deterministic graders — exact match, `contains`, structured validation, golden outputs. Those graders are fast, free, and fully reproducible. But they break down the moment the task produces *open-ended text*: two summaries can be equally faithful and concise while sharing almost no words. An `exact_match` grader marks one correct and one wrong based purely on whether the string matches a reference. A `contains` grader tells us a keyword appears, not whether the output is actually good.

LLM-as-judge plugs that gap: we ask a language model to score an output against an explicit rubric, returning a structured score and a rationale we can read. This notebook introduces the technique, shows you how to wire it into the harness from notebook 03 as just another grader, and — critically — shows you where it goes wrong and how to check whether your judge can be trusted.

This is the **first notebook in the series that requires an OpenAI API key.** An early guard cell will stop and print instructions if the key is missing.

## What you'll learn

- Why deterministic graders mis-score open-ended outputs, motivating the need for a judge.
- How to set `OPENAI_API_KEY` and what the guard cell does when it is absent.
- How to define `make_llm_judge(rubric, client)` — a factory that returns a grader with the same `(example, output) -> Score` signature as every other grader in this series.
- How to write a concrete rubric, run the judge on good and bad outputs, and read the `Score` with its `rationale` field.
- How to plug `make_llm_judge` into `run_eval` alongside deterministic graders.
- How to define `judge_pairwise(client, prompt, output_a, output_b)` and why relative judgments are often more reliable than absolute scores.
- The three main traps: **position bias**, **verbosity bias**, and **self-preference / non-determinism** — with a concrete demonstration of position bias.
- How to validate the judge against a small human-labeled set and compute an agreement rate, so the judge is not just another untested component.

## 1. Setup + API Key Guard

This notebook uses `openai` for the LLM judge. Install it if needed:

```bash
pip install openai
```

The cells below (a) import everything and re-declare the shared harness primitives inline, then (b) check for `OPENAI_API_KEY`. **If the key is missing, the guard cell prints setup instructions and raises `SystemExit` — all subsequent API-calling cells are safe to skip.**

To get an API key: visit https://platform.openai.com/api-keys, create a key, and export it in your shell before launching Jupyter:

```bash
export OPENAI_API_KEY="sk-..."
```

Anthropic users: you can swap in `anthropic` SDK calls with the same pattern — the `make_llm_judge` factory accepts any callable you pass as `client`. The default model shown here is `"gpt-4o-mini"` (inexpensive and good enough for grading).

In [1]:
# ── stdlib ────────────────────────────────────────────────────────────────────
import os
import json
from dataclasses import dataclass, field
from typing import Any, Callable

# ── Load OPENAI_API_KEY from a .env file if present; real env vars still win.
from dotenv import load_dotenv
load_dotenv()

# ── openai SDK ────────────────────────────────────────────────────────────────
from openai import OpenAI  # pip install openai

# ══════════════════════════════════════════════════════════════════════════════
# Shared harness — re-declared inline so this notebook is self-contained.
# These match the canonical definitions from notebooks 03 & 04 exactly.
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Score:
    key: str
    score: float          # normalised to [0, 1]
    passed: bool
    comment: str = ""


@dataclass
class Example:
    input: Any
    expected: Any = None
    metadata: dict = field(default_factory=dict)


@dataclass
class ExampleResult:
    example: Example
    output: Any
    scores: list[Score] = field(default_factory=list)


@dataclass
class EvalReport:
    results: list[ExampleResult]

    @property
    def pass_rate(self) -> float:
        all_scores = [s for r in self.results for s in r.scores]
        if not all_scores:
            return 0.0
        return sum(s.passed for s in all_scores) / len(all_scores)

    @property
    def mean_score(self) -> float:
        all_scores = [s for r in self.results for s in r.scores]
        if not all_scores:
            return 0.0
        return sum(s.score for s in all_scores) / len(all_scores)

    def summary_table(self) -> None:
        """Print a simple per-example summary table."""
        print(f"{'#':<4} {'passed':<8} {'mean score':<12} {'comment'}")
        print("-" * 60)
        for i, r in enumerate(self.results):
            scores = r.scores
            if not scores:
                print(f"{i:<4} {'—':<8} {'—':<12} no graders")
                continue
            passed = all(s.passed for s in scores)
            mean = sum(s.score for s in scores) / len(scores)
            comments = "; ".join(s.comment for s in scores if s.comment)
            print(f"{i:<4} {'✓' if passed else '✗':<8} {mean:<12.3f} {comments[:60]}")
        print("-" * 60)
        print(f"Pass rate: {self.pass_rate:.1%}   Mean score: {self.mean_score:.3f}")


Grader = Callable[[Example, Any], Score]


def run_eval(
    agent: Callable[[Any], Any],
    dataset: list[Example],
    graders: list[Grader],
) -> EvalReport:
    """Run `agent` on every `Example`, apply every grader, return a report."""
    results: list[ExampleResult] = []
    for example in dataset:
        output = agent(example.input)
        scores = [grader(example, output) for grader in graders]
        results.append(ExampleResult(example=example, output=output, scores=scores))
    return EvalReport(results=results)


print("Harness re-declared OK.")

Harness re-declared OK.


In [2]:
# ── API key guard ─────────────────────────────────────────────────────────────
# This cell must run before any cell that calls the OpenAI API.
# If OPENAI_API_KEY is not set, it prints setup instructions and stops the
# notebook so subsequent cells that call the API are safe to skip.

_api_key = os.getenv("OPENAI_API_KEY")

if not _api_key:
    print(
        "┌─────────────────────────────────────────────────────────────────┐\n"
        "│  OPENAI_API_KEY is not set.                                     │\n"
        "│                                                                 │\n"
        "│  This is the first notebook in the series that needs a key.    │\n"
        "│  Steps:                                                         │\n"
        "│    1. Visit https://platform.openai.com/api-keys               │\n"
        "│    2. Create a new secret key.                                  │\n"
        "│    3. In your terminal (before launching Jupyter):              │\n"
        "│         export OPENAI_API_KEY=\"sk-...\"                         │\n"
        "│    4. Restart the Jupyter kernel and re-run from the top.       │\n"
        "│                                                                 │\n"
        "│  Anthropic alternative: replace `from openai import OpenAI`    │\n"
        "│  with `import anthropic` and adapt the client calls in         │\n"
        "│  make_llm_judge / judge_pairwise to use the Messages API.       │\n"
        "└─────────────────────────────────────────────────────────────────┘"
    )
    raise SystemExit(
        "Set OPENAI_API_KEY and restart the kernel to continue."
    )

client = OpenAI()  # reads OPENAI_API_KEY from environment automatically
DEFAULT_MODEL = "gpt-4o-mini"

print(f"OpenAI client ready. Default model: {DEFAULT_MODEL}")
print("OPENAI_API_KEY found ✓")

OpenAI client ready. Default model: gpt-4o-mini
OPENAI_API_KEY found ✓


## 2. Why Deterministic Graders Fall Short

Suppose our agent summarises a paragraph. We have a reference summary. Let's define two *clearly good* candidate summaries — both faithful, concise, and complete — and watch `exact_match` and `contains` score them.

In [3]:
# ── Source paragraph and reference ────────────────────────────────────────────
SOURCE = (
    "The Apollo 11 mission, launched on July 16, 1969, successfully landed "
    "astronauts Neil Armstrong and Buzz Aldrin on the Moon on July 20. "
    "Armstrong became the first human to walk on the lunar surface, followed "
    "by Aldrin. Michael Collins orbited the Moon in the command module while "
    "his crewmates explored the surface. The crew returned safely to Earth "
    "on July 24, 1969."
)

REFERENCE = (
    "Apollo 11 (July 1969) landed Armstrong and Aldrin on the Moon; "
    "Collins orbited above. Armstrong was first to walk on the surface. "
    "The crew returned safely on July 24."
)

# Two summaries — both clearly good, written differently
SUMMARY_A = (
    "In July 1969, Apollo 11 brought Neil Armstrong and Buzz Aldrin to the "
    "lunar surface. Armstrong stepped out first. Michael Collins remained in "
    "orbit. All three astronauts returned to Earth safely on July 24th."
)

SUMMARY_B = (
    "Apollo 11 successfully landed on the Moon on July 20, 1969. "
    "Armstrong and Aldrin walked on the surface while Collins orbited. "
    "The mission concluded with a safe splashdown on July 24, 1969."
)

print("Source, reference, and two good candidate summaries defined.")

Source, reference, and two good candidate summaries defined.


In [4]:
# ── Deterministic graders ─────────────────────────────────────────────────────

def exact_match(example: Example, output: str) -> Score:
    passed = output.strip() == str(example.expected).strip()
    return Score(key="exact_match", score=1.0 if passed else 0.0, passed=passed)


def contains_keywords(keywords: list[str]) -> Grader:
    """Grader factory: passes if ALL keywords appear in the output."""
    def grader(example: Example, output: str) -> Score:
        found = [kw for kw in keywords if kw.lower() in output.lower()]
        score = len(found) / len(keywords)
        passed = score == 1.0
        comment = f"found {len(found)}/{len(keywords)}: {found}"
        return Score(key="contains_keywords", score=score, passed=passed, comment=comment)
    return grader


# Evaluate both summaries with the reference as `expected`
example_ref = Example(input=SOURCE, expected=REFERENCE)

kw_grader = contains_keywords(["Armstrong", "Aldrin", "Collins", "July 24"])

print("=== Summary A ===")
em_a = exact_match(example_ref, SUMMARY_A)
kw_a = kw_grader(example_ref, SUMMARY_A)
print(f"  exact_match  → passed={em_a.passed}, score={em_a.score}")
print(f"  contains_kw  → passed={kw_a.passed}, score={kw_a.score:.2f}, {kw_a.comment}")

print()
print("=== Summary B ===")
em_b = exact_match(example_ref, SUMMARY_B)
kw_b = kw_grader(example_ref, SUMMARY_B)
print(f"  exact_match  → passed={em_b.passed}, score={em_b.score}")
print(f"  contains_kw  → passed={kw_b.passed}, score={kw_b.score:.2f}, {kw_b.comment}")

print()
print(
    "Both summaries are accurate and well-written.\n"
    "exact_match scores both 0 (neither matches the reference string exactly).\n"
    "contains_keywords scores both 1.0 — but can't tell us HOW good they are.\n"
    "We need a grader that reads the text and reasons about quality."
)

=== Summary A ===
  exact_match  → passed=False, score=0.0
  contains_kw  → passed=True, score=1.00, found 4/4: ['Armstrong', 'Aldrin', 'Collins', 'July 24']

=== Summary B ===
  exact_match  → passed=False, score=0.0
  contains_kw  → passed=True, score=1.00, found 4/4: ['Armstrong', 'Aldrin', 'Collins', 'July 24']

Both summaries are accurate and well-written.
exact_match scores both 0 (neither matches the reference string exactly).
contains_keywords scores both 1.0 — but can't tell us HOW good they are.
We need a grader that reads the text and reasons about quality.


## 3. Rubric Grading — `make_llm_judge`

A rubric judge takes an explicit scoring criterion written in prose, sends the output (and the original input + expected reference) to a language model, and asks for a structured JSON response: `{"score": <0–1 float>, "passed": <bool>, "rationale": <string>}`. The rationale is stored in `Score.comment` so we can read *why* the judge gave each score.

`make_llm_judge` is a **factory**: it takes the rubric and configuration, and returns a grader function with the standard `(example, output) -> Score` signature — making it a drop-in for `run_eval` alongside any deterministic grader.

### Try it

In [5]:
def make_llm_judge(
    rubric: str,
    client: OpenAI,
    model: str = "gpt-4o-mini",
    key: str = "llm_judge",
) -> Grader:
    """Return a grader that scores an output against `rubric` using an LLM.

    The returned grader has signature (example, output) -> Score and can be
    passed directly to run_eval alongside deterministic graders.

    The LLM is prompted with:
      - The rubric
      - example.input (the prompt given to the agent)
      - example.expected (the reference output, if any)
      - The candidate output to score

    It must respond with JSON: {"score": <0-1 float>, "passed": <bool>,
    "rationale": <string>}. score is normalised to [0, 1] before returning.
    """

    system_prompt = (
        "You are a precise, impartial evaluator. "
        "Score the candidate output against the rubric. "
        "Respond with ONLY valid JSON matching this schema:\n"
        '{"score": <float 0-1>, "passed": <bool>, "rationale": <string>}\n'
        "No markdown fences, no extra keys."
    )

    def grader(example: Example, output: Any) -> Score:
        user_message = (
            f"## Rubric\n{rubric}\n\n"
            f"## Input given to the agent\n{example.input}\n\n"
        )
        if example.expected is not None:
            user_message += f"## Reference output\n{example.expected}\n\n"
        user_message += f"## Candidate output to score\n{output}\n\n"
        user_message += "Respond with JSON only."

        response = client.chat.completions.create(
            model=model,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},
            ],
            temperature=0,
        )

        raw = response.choices[0].message.content
        try:
            parsed = json.loads(raw)
            raw_score = float(parsed["score"])
            normalised = max(0.0, min(1.0, raw_score))  # clamp to [0, 1]
            passed = bool(parsed["passed"])
            rationale = str(parsed.get("rationale", ""))
        except (json.JSONDecodeError, KeyError, ValueError, TypeError) as exc:
            raise RuntimeError(f"LLM judge returned unexpected JSON: {raw!r}") from exc

        return Score(key=key, score=normalised, passed=passed, comment=rationale)

    return grader


print("make_llm_judge defined.")

make_llm_judge defined.


In [6]:
# ── Rubric for summary quality ─────────────────────────────────────────────────
SUMMARY_RUBRIC = (
    "Score the summary on two equally-weighted dimensions (each 0–0.5, "
    "combined into a single 0–1 score):\n"
    "  1. FAITHFULNESS: Every claim in the summary must be supported by the "
    "source text. Penalise hallucinated facts, wrong dates, or wrong names.\n"
    "  2. CONCISENESS: The summary should convey the essential information "
    "without padding or irrelevant detail.\n"
    "A score >= 0.7 should be marked passed=true. "
    "Score < 0.7 should be marked passed=false."
)

judge = make_llm_judge(rubric=SUMMARY_RUBRIC, client=client, model=DEFAULT_MODEL)

# A clearly bad summary: hallucinated detail (wrong astronaut count, wrong date)
BAD_SUMMARY = (
    "The Apollo 11 mission in August 1969 sent four astronauts to the Moon. "
    "Armstrong was the only one to walk on the surface. The crew never returned."
)

print("Scoring Summary A (good)…")
score_a = judge(Example(input=SOURCE, expected=REFERENCE), SUMMARY_A)
print(f"  key={score_a.key!r}, score={score_a.score:.3f}, passed={score_a.passed}")
print(f"  rationale: {score_a.comment[:200]}")

print()
print("Scoring Summary B (good)…")
score_b = judge(Example(input=SOURCE, expected=REFERENCE), SUMMARY_B)
print(f"  key={score_b.key!r}, score={score_b.score:.3f}, passed={score_b.passed}")
print(f"  rationale: {score_b.comment[:200]}")

print()
print("Scoring Bad Summary (hallucinated facts)…")
score_bad = judge(Example(input=SOURCE, expected=REFERENCE), BAD_SUMMARY)
print(f"  key={score_bad.key!r}, score={score_bad.score:.3f}, passed={score_bad.passed}")
print(f"  rationale: {score_bad.comment[:200]}")

Scoring Summary A (good)…


  key='llm_judge', score=0.900, passed=True
  rationale: The summary accurately reflects the key details of the Apollo 11 mission, including the launch date, the astronauts involved, their actions, and the return date. It is concise and does not include any

Scoring Summary B (good)…


  key='llm_judge', score=0.900, passed=True
  rationale: The summary accurately reflects the key events of the Apollo 11 mission, including the landing date, the astronauts involved, and the safe return date. It is concise and does not include any irrelevan

Scoring Bad Summary (hallucinated facts)…


  key='llm_judge', score=0.000, passed=False
  rationale: The summary contains multiple inaccuracies: it states the mission took place in August 1969 instead of July, claims there were four astronauts instead of three, incorrectly states that only Armstrong 


In [7]:
# ── Plug the judge into run_eval alongside a deterministic grader ──────────────

# repeated examples just to give run_eval multiple rows to display
dataset = [
    Example(input=SOURCE, expected=REFERENCE),
    Example(input=SOURCE, expected=REFERENCE),
    Example(input=SOURCE, expected=REFERENCE),
]

def summary_agent_good(inp: str) -> str:
    """Stub: always returns Summary A."""
    return SUMMARY_A

def summary_agent_bad(inp: str) -> str:
    """Stub: always returns the hallucinated bad summary."""
    return BAD_SUMMARY

graders = [
    kw_grader,    # deterministic: from section 2
    judge,        # LLM: from this section
]

print("=== Good agent ===")
report_good = run_eval(summary_agent_good, dataset, graders)
report_good.summary_table()

print()
print("=== Bad agent ===")
report_bad = run_eval(summary_agent_bad, dataset, graders)
report_bad.summary_table()

=== Good agent ===


#    passed   mean score   comment
------------------------------------------------------------
0    ✓        0.950        found 4/4: ['Armstrong', 'Aldrin', 'Collins', 'July 24']; Th
1    ✓        0.950        found 4/4: ['Armstrong', 'Aldrin', 'Collins', 'July 24']; Th
2    ✓        0.950        found 4/4: ['Armstrong', 'Aldrin', 'Collins', 'July 24']; Th
------------------------------------------------------------
Pass rate: 100.0%   Mean score: 0.950

=== Bad agent ===


#    passed   mean score   comment
------------------------------------------------------------
0    ✗        0.125        found 1/4: ['Armstrong']; The summary contains multiple inac
1    ✗        0.125        found 1/4: ['Armstrong']; The summary contains multiple inac
2    ✗        0.125        found 1/4: ['Armstrong']; The summary contains multiple inac
------------------------------------------------------------
Pass rate: 0.0%   Mean score: 0.125


## 4. Pairwise Comparison — `judge_pairwise`

Absolute rubric scores are useful, but they have a calibration problem: two different rubrics, or even two different prompts for the same rubric, can assign very different absolute values to the same output. Pairwise comparison sidesteps this: we show the judge two candidate outputs and ask only *"which is better?"* — a relative judgment that is often more reliable and robust to prompt wording.

`judge_pairwise` returns `"A"`, `"B"`, or `"tie"`.

### Try it

In [8]:
def judge_pairwise(
    client: OpenAI,
    prompt: str,
    output_a: str,
    output_b: str,
    model: str = "gpt-4o-mini",
) -> str:
    """Ask the LLM which of two outputs is better for `prompt`.

    Returns "A", "B", or "tie".

    `prompt` is the task description / input; output_a and output_b are the
    two candidates. The judge is shown both and must choose.
    """
    system_prompt = (
        "You are a fair evaluator. Given a task prompt and two candidate "
        "outputs (A and B), decide which is better.\n"
        "Respond with ONLY valid JSON: "
        '{"winner": "A" | "B" | "tie", "reason": "<one sentence>"}\n'
        "No markdown fences, no extra keys. "
        "Be objective; do not favour longer responses."
    )
    user_message = (
        f"## Task prompt\n{prompt}\n\n"
        f"## Output A\n{output_a}\n\n"
        f"## Output B\n{output_b}\n\n"
        "Which output better addresses the task? Respond with JSON only."
    )

    response = client.chat.completions.create(
        model=model,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
        temperature=0,
    )

    raw = response.choices[0].message.content
    parsed = json.loads(raw)
    winner = str(parsed.get("winner", "tie")).strip().upper()
    if winner not in {"A", "B", "TIE"}:
        winner = "TIE"
    reason = str(parsed.get("reason", ""))
    print(f"  winner={winner!r}, reason: {reason[:120]}")
    return "tie" if winner == "TIE" else winner


print("judge_pairwise defined.")

judge_pairwise defined.


In [9]:
# ── Compare Summary A vs Bad Summary ──────────────────────────────────────────
TASK_PROMPT = f"Summarise the following paragraph concisely and faithfully:\n\n{SOURCE}"

print("Comparing good Summary A vs hallucinated Bad Summary:")
winner = judge_pairwise(client, TASK_PROMPT, SUMMARY_A, BAD_SUMMARY, model=DEFAULT_MODEL)
print(f"Result: {winner!r}  (expected: 'A')\n")

print("Comparing bad Summary vs good Summary A (reversed):")
winner_rev = judge_pairwise(client, TASK_PROMPT, BAD_SUMMARY, SUMMARY_A, model=DEFAULT_MODEL)
print(f"Result: {winner_rev!r}  (expected: 'B')\n")

print("Comparing two good summaries (A vs B):")
winner_ab = judge_pairwise(client, TASK_PROMPT, SUMMARY_A, SUMMARY_B, model=DEFAULT_MODEL)
print(f"Result: {winner_ab!r}  (may be 'A', 'B', or 'tie' — both are good)")

Comparing good Summary A vs hallucinated Bad Summary:


  winner='A', reason: Output A accurately summarizes the key details of the Apollo 11 mission, while Output B contains factual inaccuracies.
Result: 'A'  (expected: 'A')

Comparing bad Summary vs good Summary A (reversed):


  winner='B', reason: Output B accurately summarizes the key details of the Apollo 11 mission, including the dates, astronauts, and their acti
Result: 'B'  (expected: 'B')

Comparing two good summaries (A vs B):


  winner='B', reason: Output B provides a more direct summary of the key events and dates from the original paragraph.
Result: 'B'  (may be 'A', 'B', or 'tie' — both are good)


## 5. The Traps

The LLM judge is a powerful tool, but it has well-documented failure modes. Before shipping a judge-based eval pipeline, you should know what these are and how to test for them.

> **Gotcha:** An LLM judge can produce confidently wrong scores. Until you validate it against human labels (section 6), it is just another untested component in your system.

### 5.1 Position Bias

Studies have shown that LLM judges tend to favour the candidate that appears **first** (position A) when two outputs are close in quality — not because it is better, but because of the model's positional preference in context. Concretely: you can sometimes flip the winner just by swapping A and B.

**How to test for it:** run the same pair twice with the slots swapped, then translate each "A"/"B" label back to its actual content. If the same content wins in both orderings, the judge is content-consistent and no position bias is evident. If the same *slot* wins regardless of what content is in it, position bias is present.

**When it appears:** position bias is most pronounced on near-equal pairs. When one summary is clearly better, a well-calibrated judge should pick it regardless of position. Run the swap test on your hardest, most ambiguous pairs to stress-test the judge.

### 5.2 Verbosity Bias

LLMs often rate **longer** outputs as better, even when the extra length is padding. Conciseness in a rubric needs to be stated explicitly and forcefully to counteract this tendency.

### 5.3 Self-Preference and Non-Determinism

A judge based on GPT-4o-mini may subtly prefer outputs that *sound* like GPT-4o-mini output (a form of self-preference). Additionally, even with `temperature=0`, the judge's scores can vary slightly across runs due to sampling non-determinism in the API infrastructure. Always validate over multiple runs before drawing conclusions.

### Try it — position bias demonstration

In [10]:
# ── Position bias: construct a close pair and swap A / B ──────────────────────
# Summary A and Summary B are both good. We run pairwise 4 times in each
# ordering to see whether the winner is stable or flips.
#
# IMPORTANT: the labels returned ("A" / "B") refer to the SLOT in that trial,
# not to the summary by content. In the A-first ordering slot A = Summary A,
# slot B = Summary B. In the B-first ordering the slots are swapped:
# slot A = Summary B, slot B = Summary A.
# We translate each label back to its content winner before tallying.

import collections

TRIALS = 4

results_ab: list[str] = []
results_ba: list[str] = []

print(f"Running {TRIALS} trials of A vs B (Summary A in slot A, Summary B in slot B) …")
for i in range(TRIALS):
    r = judge_pairwise(client, TASK_PROMPT, SUMMARY_A, SUMMARY_B, model=DEFAULT_MODEL)
    results_ab.append(r)

print(f"\nRunning {TRIALS} trials of B vs A (Summary B in slot A, Summary A in slot B) …")
for i in range(TRIALS):
    r = judge_pairwise(client, TASK_PROMPT, SUMMARY_B, SUMMARY_A, model=DEFAULT_MODEL)
    results_ba.append(r)

counts_ab = collections.Counter(results_ab)
counts_ba = collections.Counter(results_ba)

# ── Translate slot labels → content winners ────────────────────────────────────
# A-first ordering: slot A → Summary A, slot B → Summary B
content_ab = ["Summary A" if label == "A" else ("Summary B" if label == "B" else "tie")
              for label in results_ab]
# B-first ordering (swapped): slot A → Summary B, slot B → Summary A
content_ba = ["Summary B" if label == "A" else ("Summary A" if label == "B" else "tie")
              for label in results_ba]

content_counts_ab = collections.Counter(content_ab)
content_counts_ba = collections.Counter(content_ba)

# Determine the modal content winner for each ordering (ignore ties for comparison)
def modal_content_winner(counter):
    non_tie = {k: v for k, v in counter.items() if k != "tie"}
    if not non_tie:
        return "tie"
    return max(non_tie, key=non_tie.get)

winner_in_ab = modal_content_winner(content_counts_ab)
winner_in_ba = modal_content_winner(content_counts_ba)

print("\n── Raw slot results ─────────────────────────────────────────────────")
print(f"A-first ordering  → slot counts: {dict(counts_ab)}")
print(f"B-first ordering  → slot counts: {dict(counts_ba)}")

print("\n── Content-winner results (slot labels translated back to summaries) ─")
print(f"A-first ordering  → content winner: {dict(content_counts_ab)}")
print(f"B-first ordering  → content winner: {dict(content_counts_ba)}")

print("\n── Interpretation ───────────────────────────────────────────────────")
if winner_in_ab == winner_in_ba:
    print(
        f"The same summary ({winner_in_ab}) won in BOTH orderings.\n"
        "The judge was CONTENT-CONSISTENT in this run — it picked the same\n"
        "content regardless of which slot it appeared in. No position bias\n"
        "is evident here.\n"
        "\n"
        "Note: position bias is most likely to appear on near-equal pairs\n"
        "where neither summary is clearly better. When one summary is\n"
        "noticeably stronger, a well-calibrated judge should be content-\n"
        "consistent. Try a truly 50/50 pair to stress-test position bias."
    )
else:
    print(
        f"Different content won across orderings: A-first→{winner_in_ab}, "
        f"B-first→{winner_in_ba}.\n"
        "The judge's choice CHANGED depending on which slot the summary\n"
        "occupied — this is position bias at work. The judge is not just\n"
        "evaluating content; the order of presentation is influencing it."
    )

Running 4 trials of A vs B (Summary A in slot A, Summary B in slot B) …


  winner='B', reason: Output B provides a more direct summary of the key events and dates from the original paragraph.


  winner='B', reason: Output B provides a more direct summary of the key events and dates from the original paragraph.


  winner='B', reason: Output B provides a more direct summary of the key events and dates from the original paragraph.


  winner='B', reason: Output B provides a more direct summary of the key events and dates from the original paragraph.

Running 4 trials of B vs A (Summary B in slot A, Summary A in slot B) …


  winner='A', reason: Output A provides a more concise summary while maintaining key details about the mission.


  winner='A', reason: Output A provides a more concise summary while retaining all key details of the mission.


  winner='A', reason: Output A provides a more concise summary while maintaining key details about the mission.


  winner='A', reason: Output A provides a more concise summary while maintaining key details of the mission.

── Raw slot results ─────────────────────────────────────────────────
A-first ordering  → slot counts: {'B': 4}
B-first ordering  → slot counts: {'A': 4}

── Content-winner results (slot labels translated back to summaries) ─
A-first ordering  → content winner: {'Summary B': 4}
B-first ordering  → content winner: {'Summary B': 4}

── Interpretation ───────────────────────────────────────────────────
The same summary (Summary B) won in BOTH orderings.
The judge was CONTENT-CONSISTENT in this run — it picked the same
content regardless of which slot it appeared in. No position bias
is evident here.

Note: position bias is most likely to appear on near-equal pairs
where neither summary is clearly better. When one summary is
noticeably stronger, a well-calibrated judge should be content-
consistent. Try a truly 50/50 pair to stress-test position bias.


In [11]:
# ── Verbosity bias: observe scores on padded vs concise output ─────────────────

PADDED_SUMMARY = (
    "This is a summary of the text you provided above. "
    "The text discusses the Apollo 11 mission. "
    "Apollo 11 was launched in July 1969. "
    "The astronauts Neil Armstrong and Buzz Aldrin landed on the Moon on July 20. "
    "They walked on the surface. Michael Collins was in the command module orbiting. "
    "The mission was a success. The crew came back to Earth on July 24, 1969. "
    "This was a historic event in human spaceflight history and mankind as a whole. "
    "It demonstrated the capability of the United States space program at the time. "
    "In conclusion, Apollo 11 was a very significant mission."
)

judge_verbose = make_llm_judge(
    rubric=SUMMARY_RUBRIC,
    client=client,
    model=DEFAULT_MODEL,
    key="llm_judge_padded",
)

# re-scored here as a standalone baseline for the comparison
print("Scoring concise Summary B:")
score_concise = judge_verbose(Example(input=SOURCE, expected=REFERENCE), SUMMARY_B)
print(f"  score={score_concise.score:.3f}, passed={score_concise.passed}")
print(f"  rationale: {score_concise.comment[:200]}")

print()
print("Scoring padded summary (same facts, much more verbose):")
score_padded = judge_verbose(Example(input=SOURCE, expected=REFERENCE), PADDED_SUMMARY)
print(f"  score={score_padded.score:.3f}, passed={score_padded.passed}")
print(f"  rationale: {score_padded.comment[:200]}")

print()
print(
    "If the padded summary scores >= the concise one, verbosity bias may be\n"
    "present. The rubric explicitly penalises padding — check whether the\n"
    "judge's rationale actually mentions it."
)

Scoring concise Summary B:


  score=0.900, passed=True
  rationale: The summary accurately reflects the key details of the Apollo 11 mission, including the landing date, the astronauts involved, and the safe return date. It is concise and does not include any irreleva

Scoring padded summary (same facts, much more verbose):


  score=0.500, passed=False
  rationale: The summary contains accurate information but includes unnecessary details and subjective statements, such as 'historic event in human spaceflight history' and 'very significant mission,' which detrac

If the padded summary scores >= the concise one, verbosity bias may be
present. The rubric explicitly penalises padding — check whether the
judge's rationale actually mentions it.


## 6. Evaluating the Judge

The key insight from this notebook: **an unvalidated judge is just another untested component.** If the judge gives wrong scores, every eval pipeline downstream of it is wrong too — silently.

The cheapest validation is a small **human-labeled set**: a handful of (input, output) pairs where a human has already decided the correct score or preference. We run the judge on the same pairs and compute an **agreement rate**: what fraction of the judge's pass/fail decisions match the human's?

A real-world threshold depends on your use case, but as a rule of thumb: below ~70% agreement on a balanced set, the judge is adding noise rather than signal.

### Try it

In [12]:
# ── Human-labeled validation set (defined inline) ─────────────────────────────
# Each entry: (source_text, candidate_summary, human_passed: bool)
# These represent a human reviewer's ground-truth judgments.

HUMAN_LABELED = [
    # (input, candidate, human_passed)
    (
        SOURCE,
        SUMMARY_A,
        True,   # human: good — accurate and concise
    ),
    (
        SOURCE,
        SUMMARY_B,
        True,   # human: good — accurate and concise
    ),
    (
        SOURCE,
        BAD_SUMMARY,
        False,  # human: fail — wrong month, wrong crew count, wrong outcome
    ),
    (
        SOURCE,
        (
            "Neil Armstrong walked on the Moon. "
            "The mission was called Apollo. "
            "There were some other astronauts too."
        ),
        False,  # human: fail — vague, missing key facts (dates, Collins role)
    ),
]

print(f"Human-labeled validation set: {len(HUMAN_LABELED)} examples")
for i, (src, summ, human) in enumerate(HUMAN_LABELED):
    print(f"  [{i}] human_passed={human}  summary[:60]: {summ[:60]!r}")

Human-labeled validation set: 4 examples
  [0] human_passed=True  summary[:60]: 'In July 1969, Apollo 11 brought Neil Armstrong and Buzz Aldr'
  [1] human_passed=True  summary[:60]: 'Apollo 11 successfully landed on the Moon on July 20, 1969. '
  [2] human_passed=False  summary[:60]: 'The Apollo 11 mission in August 1969 sent four astronauts to'
  [3] human_passed=False  summary[:60]: 'Neil Armstrong walked on the Moon. The mission was called Ap'


In [13]:
# ── Run the judge on each example and compare to human label ──────────────────

agreement_judge = make_llm_judge(rubric=SUMMARY_RUBRIC, client=client, model=DEFAULT_MODEL, key="llm_judge")

agreements = []
print(f"{'#':<4} {'human':<8} {'judge':<8} {'agree':<8} rationale[:60]")
print("-" * 70)

for i, (src, summ, human_passed) in enumerate(HUMAN_LABELED):
    score = agreement_judge(Example(input=src, expected=REFERENCE), summ)
    agree = score.passed == human_passed
    agreements.append(agree)
    print(
        f"{i:<4} {str(human_passed):<8} {str(score.passed):<8} "
        f"{'✓' if agree else '✗':<8} {score.comment[:60]}"
    )

agreement_rate = sum(agreements) / len(agreements)
print("-" * 70)
print(f"Agreement rate: {agreement_rate:.1%}  ({sum(agreements)}/{len(agreements)} examples)")
print()
if agreement_rate >= 0.75:
    print("Judge agrees with human labels at >= 75% — reasonable signal for this rubric.")
else:
    print(
        "Agreement below 75% — consider revising the rubric or expanding the "
        "human-labeled set before trusting this judge in a pipeline."
    )

#    human    judge    agree    rationale[:60]
----------------------------------------------------------------------


0    True     True     ✓        The summary accurately reflects the key details of the Apoll


1    True     True     ✓        The summary accurately reflects the key details of the Apoll


2    False    False    ✓        The summary contains multiple inaccuracies: it states the mi


3    False    False    ✓        The summary lacks key details such as the mission name (Apol
----------------------------------------------------------------------
Agreement rate: 100.0%  (4/4 examples)

Judge agrees with human labels at >= 75% — reasonable signal for this rubric.


## What you just learned

- **Deterministic graders break on open-ended text** — `exact_match` marks good summaries wrong; `contains` can't distinguish good from bad.
- **`make_llm_judge(rubric, client)`** returns a grader with the standard `(example, output) -> Score` signature. It prompts the model with the rubric + context, parses a JSON response into a `Score`, and stores the rationale in `comment`. It plugs into `run_eval` as a drop-in alongside deterministic graders.
- **`judge_pairwise(client, prompt, output_a, output_b)`** returns `"A"`, `"B"`, or `"tie"`. Relative judgments are often more reliable than absolute rubric scores for close-quality pairs.
- **Position bias** surfaces on near-equal pairs: the judge may favour whichever candidate appears in the first slot, not because it is better. To test for it, run the swap experiment and translate labels back to content — if the same content wins in both orderings the judge is content-consistent; if the same *slot* wins regardless, bias is present. This run showed a content-consistent judge.
- **Verbosity bias**: LLMs tend to rate longer outputs higher unless the rubric explicitly penalises padding.
- **Self-preference and non-determinism**: a GPT-4o-mini judge may subtly prefer GPT-like output, and scores can shift across runs even at `temperature=0`. Validate over multiple runs before trusting aggregate metrics.
- **An unvalidated judge is just another untested component.** Validate against a small human-labeled set and compute an agreement rate before trusting the judge in a pipeline.

## What's missing

We can now score outputs — deterministically and with an LLM judge. But we still can't **see** what a multi-step agent did internally while producing those outputs. When a score is wrong, is the agent's first step failing? Its tool call? Its final synthesis? We have no visibility.

In **notebook 06 — `06_tracing_with_langsmith.ipynb`** we instrument a real multi-step agent, run it, and read its trace in the LangSmith UI: every span, every intermediate input and output, latency per step, and token usage. That's the observability layer that makes debugging and root-cause analysis possible.